In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import pandas as pd
import datetime as dt

In [3]:
from rockyelevate.wrapper import Session
from rockyelevate.utils import response_to_dataframe as elv_res_to_df

In [4]:
elv = Session("PROD", multithread=True, max_threads=40)

In [5]:
all_orgs = elv.get_orgs_by_id()
all_orgs_df = elv_res_to_df(all_orgs)
org_ids = list(all_orgs_df['id'].unique())

In [6]:
plan_years = elv.get_plan_years(oids=org_ids)
pydf = elv_res_to_df(plan_years)

In [31]:
plan_year_df= pydf.copy()

for col in ["valid_from", "valid_to"]:
    plan_year_df[col] = pd.to_datetime(plan_year_df[col])


In [46]:
NOW = dt.datetime.now()


plan_year_df= pydf.copy()

for col in ["valid_from", "valid_to"]:
    plan_year_df[col] = pd.to_datetime(plan_year_df[col])

# add length of plan (days)
plan_year_df["plan_year_length"] = plan_year_df.apply(lambda row: (row["valid_to"] - row["valid_from"]), axis=1)

# add bool column saying if this plan year is current
plan_year_df["is_current"] = plan_year_df.apply(lambda row: row["valid_from"] < NOW < row["valid_to"], axis=1)

# add bool column saying if this plan year was the plan year before the current one
current_prior_ids = plan_year_df[plan_year_df["is_current"]]["prior_plan_year.id"]
plan_year_df["is_prior"] = plan_year_df["id"].isin(current_prior_ids)

short_plan = plan_year_df['plan_year_length'] < pd.Timedelta(days=363)

short_current = set(plan_year_df[plan_year_df['is_current'] & short_plan]['organization_id'])
short_prior = set(plan_year_df[plan_year_df['is_prior'] & short_plan]['organization_id'])

consecutive_short_orgs = short_current & short_prior

# report_df = plan_year_df[plan_year_df['organization_id'].isin(consecutive_short_orgs)][
#     ['organization_id', 'name', 'valid_from', 'valid_to', 'plan_year_length', 'is_current', 'is_prior']
# ].sort_values(['organization_id', 'valid_from'])

# report_df

consecutive_short_orgs


{5285, 8895, 8920, 9556, 9768, 16256, 21607, 28072}

In [ ]:
NOW = dt.datetime.now()

# add length of plan (days)
plan_year_df["plan_year_length"] = plan_year_df.apply(lambda row: (row["valid_to"] - row["valid_from"]), axis=1)

# add bool column saying if this plan year is current
plan_year_df["is_current"] = plan_year_df.apply(lambda row: row["valid_from"] < NOW < row["valid_to"], axis=1)

# add bool column saying if this plan year was the plan year before the current one
current_prior_ids = plan_year_df[plan_year_df["is_current"]]["prior_plan_year.id"]
plan_year_df["is_prior"] = plan_year_df["id"].isin(current_prior_ids)

plan_year_df = plan_year_df[plan_year_df["is_current"] | plan_year_df["is_prior"]]

short_plan = (plan_year_df['plan_year_length'] < pd.Timedelta(days=363))

current_short_orgs = plan_year_df[plan_year_df['is_current'] & short_plan]['organization_id']
prior_short_orgs = plan_year_df[plan_year_df['is_prior'] & short_plan]['organization_id']

short_org_ids = pd.concat([current_short_orgs, prior_short_orgs]).unique()

In [35]:
short_org_ids

array([ 8519, 12783, 35594, 35730, 35018, 11859, 12128,  8552,  8554,
        8535,  8499,  8493,  8488,  8487,  9989,  9977,  9951,  9091,
        9019,  9020,  9021,  9000,  8998, 29702,  8832,  8823,  8802,
        8793,  8779, 10863, 10621, 10624, 10574, 10532, 10555, 10529,
        8625,  8626,  8622,  8619,  8608,  8599,  8582,  8581,  8583,
       10306,  9919,  9895,  9873, 10100, 10089, 10081,  8967,  8963,
        8958,  8953,  8944,  8931,  8923,  8920,  9850,  9845,  9821,
        9824,  9802, 28072, 27989, 26920, 26830,  8145,  6223,  6497,
        6125,  6117,  6114,  6112,  6104,  6100,  6095,  6092,  6094,
        6057,  8685,  8677,  8665,  8655,  8646,  8645,  8641,  8637,
        8632,  8900,  8896,  8895,  8875,  8858,  8845,  8843, 22256,
        5849,  5720,  5285,  5373, 23152, 22293, 22707, 21607, 16256,
       25549, 25241, 25230, 25064, 23840, 23671, 23416,  9722,  9610,
        9590,  9556,  9534,  8764,  8730,  8726,  8706,  8697, 10069,
       10066,  9777,

In [ ]:

plan_year_df["plan_year_length"] = plan_year_df.apply(lambda row: (row["valid_to"] - row["valid_from"]), axis=1)
plan_year_df["is_current"] = plan_year_df.apply(lambda row: row["valid_from"] < NOW < row["valid_to"], axis=1)

current_prior_ids = plan_year_df[plan_year_df["is_current"]]["prior_plan_year.id"]
plan_year_df["is_prior"] = plan_year_df["id"].isin(current_prior_ids)

plan_year_df = plan_year_df[plan_year_df["is_current"] | plan_year_df["is_prior"]]

plan_year_df

,id,organization_id,name,valid_from,valid_to,organization_path,prior_plan_year_id,prior_plan_year.id,prior_plan_year.name,prior_plan_year.valid_from,prior_plan_year.valid_to,plan_year_length,is_current,is_prior
1,40907,8719,01/01/2025 - 12/31/2025,2025-01-01,2025-12-31,\0\4928\5012\8719\,32223.0,32223.0,01/01/2024 - 12/31/2024,01/01/2024,12/31/2024,364 days,False,True
2,91678,8719,01/01/2026 - 12/31/2026_94254583,2026-01-01,2026-12-31,\0\4928\5012\8719\,40907.0,40907.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,364 days,True,False
4,40678,8494,01/01/2025 - 12/31/2025,2025-01-01,2025-12-31,\0\4928\5012\8494\,31953.0,31953.0,01/01/2024 - 12/31/2024,01/01/2024,12/31/2024,364 days,False,True
5,91479,8494,01/01/2026 - 12/31/2026_57f54aaa,2026-01-01,2026-12-31,\0\4928\5012\8494\,40678.0,40678.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,364 days,True,False
9,41086,8918,01/01/2025 - 12/31/2025,2025-01-01,2025-12-31,\0\4928\5012\8918\,32445.0,32445.0,01/01/2024 - 12/31/2024,01/01/2024,12/31/2024,364 days,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4113,91513,8534,01/01/2026 - 12/31/2026_257e3b7e,2026-01-01,2026-12-31,\0\4928\5012\8534\,40717.0,40717.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,364 days,True,False
4115,40710,8527,01/01/2025 - 12/31/2025,2025-01-01,2025-12-31,\0\4928\5012\8527\,31991.0,31991.0,01/01/2024 - 12/31/2024,01/01/2024,12/31/2024,364 days,False,True
4116,91508,8527,01/01/2026 - 12/31/2026_15e6e173,2026-01-01,2026-12-31,\0\4928\5012\8527\,40710.0,40710.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,364 days,True,False
4124,31166,7042,07/01/2024 - 06/30/2025,2024-07-01,2025-06-30,\0\4928\5012\7042\,NaN,NaN,NaN,NaN,NaN,364 days,False,True


In [30]:
plan_year_df['current_short'] = plan_year_df.apply(lambda row: row['is_current'] & (row['plan_year_length'] < pd.Timedelta(days=363)), axis=1)
plan_year_df['prior_short'] = plan_year_df.apply(lambda row: row['is_prior'] & (row['plan_year_length'] < pd.Timedelta(days=363)), axis=1)

plan_year_df['has_short'] = plan_year_df.apply(lambda row: row['current_short'] | row['prior_short'], axis=1)
plan_year_df[plan_year_df['has_short']]

C:\Users\james.richmond\AppData\Local\Temp\ipykernel_24236\3983872773.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plan_year_df['current_short'] = plan_year_df.apply(lambda row: row['is_current'] & (row['plan_year_length'] < pd.Timedelta(days=363)), axis=1)
C:\Users\james.richmond\AppData\Local\Temp\ipykernel_24236\3983872773.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plan_year_df['prior_short'] = plan_year_df.apply(lambda row: row['is_prior'] & (row['plan_year_length'] < pd.Timedelta(days=3

,id,organization_id,name,valid_from,valid_to,organization_path,prior_plan_year_id,prior_plan_year.id,prior_plan_year.name,prior_plan_year.valid_from,prior_plan_year.valid_to,plan_year_length,is_current,is_prior,current_short,prior_short,has_short
40,91500,8519,01/01/2026 - 03/31/2026,2026-01-01,2026-03-31,\0\4928\5012\8519\,40702.0,40702.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,89 days,True,False,True,False,True
86,110728,12783,02/01/2026 - 12/31/2026,2026-02-02,2026-12-31,\0\4928\5012\12783\,NaN,NaN,NaN,NaN,NaN,332 days,True,False,True,False,True
87,113203,35594,03/01/2026-12/31/2026,2026-03-01,2026-12-31,\0\4928\5012\35594\,NaN,NaN,NaN,NaN,NaN,305 days,True,False,True,False,True
88,113506,35730,03/01/2026-12/31/2026,2026-03-01,2026-12-31,\0\4928\5012\35730\,NaN,NaN,NaN,NaN,NaN,305 days,True,False,True,False,True
92,113503,35018,03/01/2026 - 12/31/2026,2026-03-01,2026-12-31,\0\4928\5012\35018\,NaN,NaN,NaN,NaN,NaN,305 days,True,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3840,106576,9777,01/01/2026 - 09/30/2026,2026-01-01,2026-09-30,\0\4928\5012\9777\,86305.0,86305.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,272 days,True,False,True,False,True
3881,106486,9768,01/01/2026 - 12/15/2026,2026-01-01,2026-12-15,\0\4928\5012\9768\,41261.0,41261.0,01/01/2025 - 12/15/2025,01/01/2025,12/15/2025,348 days,True,False,True,False,True
3884,41261,9768,01/01/2025 - 12/15/2025,2025-01-01,2025-12-15,\0\4928\5012\9768\,33489.0,33489.0,01/01/2024 - 12/15/2024,01/01/2024,12/15/2024,348 days,False,True,False,True,True
4019,91851,8927,01/01/2026 - 09/30/2026,2026-01-01,2026-09-30,\0\4928\5012\8927\,41095.0,41095.0,01/01/2025 - 12/31/2025,01/01/2025,12/31/2025,272 days,True,False,True,False,True


In [ ]:
id,
organization_id,
name,
valid_from,
valid_to,
organization_path,
prior_plan_year_id,
prior_plan_year.id,
prior_plan_year.name,
prior_plan_year.valid_from,
prior_plan_year.valid_to,
plan_year_length,
is_current,
is_prior